In [1]:
import pandas as pd
from pathlib import Path

DATA = Path('../data')

## HPC HNO — Cluster column

In [2]:
hno_2024 = pd.read_csv(DATA / 'hpc_hno_2024.csv', skiprows=[1])  # skip HXL tag row
hno_2025 = pd.read_csv(DATA / 'hpc_hno_2025.csv', skiprows=[1])
hno_2026 = pd.read_csv(DATA / 'hpc_hno_2026.csv')

hno_clusters = (
    pd.concat([
        hno_2024[['Cluster']].assign(source='hpc_hno_2024'),
        hno_2025[['Cluster']].assign(source='hpc_hno_2025'),
        hno_2026[['Cluster']].assign(source='hpc_hno_2026'),
    ])
    .dropna(subset=['Cluster'])
    .groupby('Cluster')['source'].apply(lambda s: ', '.join(sorted(s.unique())))
    .reset_index()
    .rename(columns={'source': 'present_in'})
    .sort_values('Cluster')
    .reset_index(drop=True)
)

print(f'{len(hno_clusters)} unique clusters across HNO 2024/2025/2026')
hno_clusters

/tmp/ipykernel_150848/1507177348.py:1: DtypeWarning: Columns (0: Admin 1 PCode, 1: Admin 1 Name, 2: Admin 2 PCode, 3: Admin 2 Name, 4: Admin 3 PCode, 5: Admin 3 Name, 6: Info) have mixed types. Specify dtype option on import or set low_memory=False.
  hno_2024 = pd.read_csv(DATA / 'hpc_hno_2024.csv', skiprows=[1])  # skip HXL tag row


/tmp/ipykernel_150848/1507177348.py:2: DtypeWarning: Columns (0: Admin 1 PCode, 1: Admin 1 Name, 2: Admin 3 PCode, 3: Admin 3 Name, 4: Info) have mixed types. Specify dtype option on import or set low_memory=False.
  hno_2025 = pd.read_csv(DATA / 'hpc_hno_2025.csv', skiprows=[1])


20 unique clusters across HNO 2024/2025/2026


,Cluster,present_in
0,AGR,hpc_hno_2024
1,ALL,"hpc_hno_2024, hpc_hno_2025, hpc_hno_2026"
2,CCM,"hpc_hno_2024, hpc_hno_2025, hpc_hno_2026"
3,CSS,"hpc_hno_2024, hpc_hno_2025"
4,EDU,"hpc_hno_2024, hpc_hno_2025, hpc_hno_2026"
5,ERY,"hpc_hno_2024, hpc_hno_2025"
6,FSC,"hpc_hno_2024, hpc_hno_2025, hpc_hno_2026"
7,HEA,"hpc_hno_2024, hpc_hno_2025, hpc_hno_2026"
8,LOG,"hpc_hno_2024, hpc_hno_2025"
9,MPC,"hpc_hno_2025, hpc_hno_2026"


## FTS Requirements — cluster & globalcluster

In [3]:
fts_globalcluster = pd.read_csv(DATA / 'fts_requirements_funding_globalcluster_global.csv')

fts_clusters = (
    pd.concat([
        fts_globalcluster[['cluster']].assign(source='fts_globalcluster'),
    ])
    .dropna(subset=['cluster'])
    .drop_duplicates(subset=['cluster'])
    .reset_index()
    .sort_values('cluster')
    .reset_index(drop=True)
)

print(f'{len(fts_clusters)} unique clusters in FTS requirements')
fts_clusters

24 unique clusters in FTS requirements


,index,cluster,source
0,14,Agriculture,fts_globalcluster
1,121,COVID-19,fts_globalcluster
2,656,Camp Coordination / Management,fts_globalcluster
3,0,Coordination and support services,fts_globalcluster
4,16,Early Recovery,fts_globalcluster
5,1,Education,fts_globalcluster
6,2,Emergency Shelter and NFI,fts_globalcluster
7,246,Emergency Telecommunications,fts_globalcluster
8,3,Food Security,fts_globalcluster
9,4,Health,fts_globalcluster


## Project Summary — Cluster column

In [4]:
projects = pd.read_csv(DATA / 'ProjectSummaryWithLocationAndCluster20260418111139715.csv')

project_clusters = (
    projects['Cluster']
    .dropna()
    .drop_duplicates()
    .sort_values()
    .reset_index(drop=True)
    .to_frame()
)

print(f'{len(project_clusters)} unique clusters in Project Summary')
project_clusters

21 unique clusters in Project Summary


,Cluster
0,COORDINATION AND COMMON SERVICES
1,Coordination and Support Services
2,EDUCATION
3,EMERGENCY SHELTER AND NON-FOOD ITEMS
4,Education
5,Emergency Shelter and NFI
6,FOOD SECURITY AND AGRICULTURE
7,Food Security
8,HEALTH
9,Health


## Cross-file cluster overlap

## HNO code → FTS globalcluster mapping

In [5]:
HNO_TO_FTS = {
    'AGR':     'Agriculture',
    'ALL':     None,  # cross-cutting / all clusters
    'CCM':     'Camp Coordination / Management',
    'CSS':     'Coordination and support services',
    'EDU':     'Education',
    'ERY':     'Early Recovery',
    'FSC':     'Food Security',
    'HEA':     'Health',
    'LOG':     'Logistics',
    'MPC':     'Multipurpose Cash',
    'MS':      'Multi-sector',
    'NUT':     'Nutrition',
    'PRO':     'Protection',
    'PRO-CPN': 'Protection - Child Protection',
    'PRO-GBV': 'Protection - Gender-Based Violence',
    'PRO-HLP': 'Protection - Housing, Land and Property',
    'PRO-MIN': 'Protection - Mine Action',
    'SHL':     'Emergency Shelter and NFI',
    'TEL':     'Emergency Telecommunications',
    'WSH':     'Water Sanitation Hygiene',
}

mapping_df = (
    pd.DataFrame.from_dict(HNO_TO_FTS, orient='index', columns=['fts_globalcluster'])
    .rename_axis('hno_code')
    .reset_index()
    .sort_values('hno_code')
    .reset_index(drop=True)
)

fts_names = set(fts_clusters['cluster'])
mapping_df['fts_match'] = mapping_df['fts_globalcluster'].apply(
    lambda x: x in fts_names if x else None
)

print(f"Codes with no FTS match: {mapping_df[mapping_df['fts_match'] == False]['hno_code'].tolist()}")
mapping_df

Codes with no FTS match: ['ALL']


,hno_code,fts_globalcluster,fts_match
0,AGR,Agriculture,True
1,ALL,NaN,False
2,CCM,Camp Coordination / Management,True
3,CSS,Coordination and support services,True
4,EDU,Education,True
5,ERY,Early Recovery,True
6,FSC,Food Security,True
7,HEA,Health,True
8,LOG,Logistics,True
9,MPC,Multipurpose Cash,True


In [6]:
fts_names = set(fts_clusters['cluster'])
hno_mapped = {HNO_TO_FTS[c] for c in hno_clusters['Cluster'] if HNO_TO_FTS.get(c)}
project_names = set(project_clusters['Cluster'])

all_clusters = sorted(fts_names | hno_mapped | project_names)

overlap = pd.DataFrame({
    'fts_globalcluster': all_clusters,
    'in_hno': [c in hno_mapped for c in all_clusters],
    'in_fts': [c in fts_names for c in all_clusters],
    'in_projects': [c in project_names for c in all_clusters],
})
overlap['hno_code'] = overlap['fts_globalcluster'].map({v: k for k, v in HNO_TO_FTS.items() if v})
overlap['count'] = overlap[['in_hno', 'in_fts', 'in_projects']].sum(axis=1)
overlap = overlap.sort_values(['count', 'fts_globalcluster'], ascending=[False, True]).reset_index(drop=True)

print(f'{len(overlap)} unique clusters total (normalized to FTS globalcluster names)')
overlap

39 unique clusters total (normalized to FTS globalcluster names)


,fts_globalcluster,in_hno,in_fts,in_projects,hno_code,count
0,Education,True,True,True,EDU,3
1,Emergency Shelter and NFI,True,True,True,SHL,3
2,Food Security,True,True,True,FSC,3
3,Health,True,True,True,HEA,3
4,Nutrition,True,True,True,NUT,3
5,Protection,True,True,True,PRO,3
6,Agriculture,True,True,False,AGR,2
7,Camp Coordination / Management,True,True,False,CCM,2
8,Coordination and support services,True,True,False,CSS,2
9,Early Recovery,True,True,False,ERY,2
